# E-Commerce Customer Churn Prevention — EDA & Model Training

This notebook walks through the full analysis pipeline:
1. **Data Loading & Validation** — Load and inspect the customer churn features dataset
2. **Exploratory Data Analysis** — 8 visualizations uncovering key churn drivers
3. **XGBoost Model Training** — Train, evaluate, and extract feature importances
4. **Summary** — Key findings and how they feed into the agentic retention pipeline

In [1]:
import sys
from pathlib import Path

# Allow imports from the project src/ directory
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.1)

## 1. Data Loading & Validation

In [2]:
from src.data_loader import load_dataset, validate_dataset, get_feature_groups

df = load_dataset(PROJECT_ROOT / "data" / "customer_churn_features.csv")
summary = validate_dataset(df)

print(f"Rows: {summary['n_rows']:,}")
print(f"Columns: {summary['n_cols']}")
print(f"Churn rate: {summary['churn_rate']:.1%}")
print(f"Churn distribution: {summary['churn_distribution']}")
print(f"Missing columns: {summary['missing_columns']}")

df.head()

Rows: 36,992
Columns: 26
Churn rate: 54.1%
Churn distribution: {1: 20012, 0: 16980}
Missing columns: []


,age,gender,security_no,region_category,membership_category,joined_through_referral,preferred_offer_types,medium_of_operation,internet_option,days_since_last_login,...,past_complaint,complaint_status,feedback,churn_risk_score,tenure_days,value_segment,engagement_segment,price_sensitivity,support_risk,profile_tags
0,18,0,XW0DQ7H,2,3,0.0,1,3,2,17.0,...,0,1,4,0,136,high,low,high,0,value_high;engagement_low;price_sensitive
1,32,0,5K0N3X1,0,4,0.0,1,1,1,16.0,...,1,2,5,0,125,low,low,high,1,value_low;engagement_low;price_sensitive;suppo...
2,44,0,1F2TCL3,1,2,1.0,1,1,2,14.0,...,1,3,3,1,415,medium,medium,high,1,value_medium;engagement_medium;price_sensitive...
3,37,1,VJGJ33N,0,2,1.0,1,1,1,11.0,...,1,4,3,1,428,medium,medium,high,1,value_medium;engagement_medium;price_sensitive...
4,31,0,SVZXCWB,0,2,0.0,0,2,1,20.0,...,1,2,3,1,22,medium,low,high,1,value_medium;engagement_low;price_sensitive;su...


In [3]:
groups = get_feature_groups(df)
print("Numeric features:", groups["numeric"])
print("\nCategorical features:", groups["categorical"])

df.describe()

Numeric features: ['age', 'gender', 'region_category', 'membership_category', 'joined_through_referral', 'preferred_offer_types', 'medium_of_operation', 'internet_option', 'days_since_last_login', 'avg_time_spent', 'avg_transaction_value', 'avg_frequency_login_days', 'points_in_wallet', 'used_special_discount', 'offer_application_preference', 'past_complaint', 'complaint_status', 'feedback', 'tenure_days', 'support_risk']

Categorical features: ['security_no', 'value_segment', 'engagement_segment', 'price_sensitivity', 'profile_tags']


,age,gender,region_category,membership_category,joined_through_referral,preferred_offer_types,medium_of_operation,internet_option,days_since_last_login,avg_time_spent,...,avg_frequency_login_days,points_in_wallet,used_special_discount,offer_application_preference,past_complaint,complaint_status,feedback,churn_risk_score,tenure_days,support_risk
count,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,...,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000,36992.000000
mean,37.118161,0.501757,1.076179,2.242458,0.424822,1.010354,1.563689,1.004785,12.771599,279.147450,...,16.181973,688.492829,0.549903,0.552552,0.497135,1.625946,3.072989,0.540982,544.979915,0.497135
std,15.867412,0.503184,1.025918,1.736675,0.494323,0.830325,0.862282,0.816289,5.420212,329.947709,...,8.200415,182.060475,0.497510,0.497237,0.499999,1.216914,2.560848,0.498324,317.778300,0.499999
min,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,9.000000,60.102500,...,10.000000,624.350000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,268.000000,0.000000
50%,37.000000,1.000000,1.000000,2.000000,0.000000,1.000000,2.000000,1.000000,13.000000,161.765000,...,16.000000,697.620000,1.000000,1.000000,0.000000,1.000000,2.000000,1.000000,544.000000,0.000000
75%,51.000000,1.000000,2.000000,4.000000,1.000000,2.000000,2.000000,2.000000,16.000000,356.515000,...,22.000000,757.002500,1.000000,1.000000,1.000000,2.000000,5.000000,1.000000,822.000000,1.000000
max,64.000000,2.000000,3.000000,5.000000,1.000000,3.000000,3.000000,2.000000,26.000000,3235.578521,...,73.061995,2069.069761,1.000000,1.000000,1.000000,4.000000,8.000000,1.000000,1095.000000,1.000000


## 2. Exploratory Data Analysis

### 2.1 Churn Distribution

In [4]:
from src.eda import (
    plot_churn_distribution, plot_churn_by_membership, plot_numeric_by_churn,
    plot_correlation_heatmap, plot_churn_by_segments, plot_complaint_support_churn,
    plot_tenure_distribution, plot_churn_by_price_sensitivity,
)

fig = plot_churn_distribution(df)
plt.show()

**Finding:** 54.1% churn rate — roughly balanced, no resampling needed for ML. But losing over half your customers is a critical business problem.

### 2.2 Churn Rate by Membership Category

In [5]:
from src.preprocessing import get_churn_stats_by_column

fig = plot_churn_by_membership(df)
plt.show()

print("\nChurn rate and count by membership category:")
get_churn_stats_by_column(df, "membership_category")


Churn rate and count by membership category:


,churn_rate,count
membership_category,,
2,0.970619,7692
0,0.967504,7724
5,0.427522,5988
1,0.369831,6795
3,0.000000,4338
4,0.000000,4455


**Finding:** The single most powerful predictor. Categories 0 & 2 have ~97% churn; categories 3 & 4 have 0% churn. Membership tier almost entirely determines churn.

### 2.3 Numeric Features by Churn Status

In [6]:
from src.preprocessing import get_numeric_stats_by_churn

fig = plot_numeric_by_churn(df)
plt.show()

num_cols = ["age", "days_since_last_login", "avg_time_spent",
            "avg_transaction_value", "avg_frequency_login_days",
            "points_in_wallet", "tenure_days"]
print("\nMedian values by churn status:")
get_numeric_stats_by_churn(df, num_cols)


Median values by churn status:


,feature,retained_median,churned_median,delta
0,age,37.000,37.00,0.000
1,days_since_last_login,13.000,13.00,0.000
2,avg_time_spent,169.255,156.86,-12.395
3,avg_transaction_value,30602.620,25384.44,-5218.180
4,avg_frequency_login_days,15.000,16.00,1.000
5,points_in_wallet,749.430,647.92,-101.510
6,tenure_days,550.000,539.00,-11.000


**Finding:** Retained customers have higher transaction values ($30.6K vs $25.4K median) and more wallet points (749 vs 648). Age, tenure, and login recency show no meaningful difference.

### 2.4 Feature Correlation Heatmap

In [7]:
from src.preprocessing import compute_correlations

fig = plot_correlation_heatmap(df)
plt.show()

print("\nTop correlations with churn_risk_score:")
compute_correlations(df).head(10)


Top correlations with churn_risk_score:


membership_category             0.457035
points_in_wallet                0.295604
avg_transaction_value           0.218012
feedback                        0.207723
avg_frequency_login_days        0.132145
joined_through_referral         0.028695
preferred_offer_types           0.024618
medium_of_operation             0.018839
offer_application_preference    0.018729
days_since_last_login           0.016382
Name: churn_risk_score, dtype: float64

**Finding:** `membership_category` has the strongest correlation with churn (0.46). `support_risk` and `past_complaint` are perfectly correlated (r=1.0) — one is redundant and should be dropped.

### 2.5 Churn by Value & Engagement Segments

In [8]:
fig = plot_churn_by_segments(df)
plt.show()

for seg in ["value_segment", "engagement_segment"]:
    print(f"\nChurn by {seg}:")
    print(get_churn_stats_by_column(df, seg))


Churn by value_segment:
               churn_rate  count
value_segment                   
low              0.590105  12208
medium           0.581797  12207
high             0.453685  12577

Churn by engagement_segment:
                    churn_rate  count
engagement_segment                   
low                   0.547683  13422
medium                0.544387  11321
high                  0.530492  12249


**Finding:** Value segmentation differentiates churn (high-value = 45%, below average). Engagement segmentation barely differentiates (all ~53-55%).

### 2.6 Complaints, Support Risk & Feedback

In [9]:
fig = plot_complaint_support_churn(df)
plt.show()

print("\nChurn by feedback type:")
print(get_churn_stats_by_column(df, "feedback"))


Churn by feedback type:
          churn_rate  count
feedback                   
2           0.649921   6350
1           0.638516   6252
0           0.634976   6290
3           0.631797   6271
7           0.627011   6279
4           0.000000   1382
5           0.000000   1360
6           0.000000   1417
8           0.000000   1391


**Finding:** Feedback is a near-perfect churn signal: types 4-6, 8 have 0% churn; types 0-3, 7 have 63-65% churn. Past complaints have surprisingly small effect (+1.7%).

### 2.7 Tenure Distribution

In [10]:
fig = plot_tenure_distribution(df)
plt.show()

**Finding:** Churn is spread uniformly across all tenure levels — this is NOT a lifecycle problem. Long-tenured customers are not meaningfully safer.

### 2.8 Price Sensitivity

In [11]:
fig = plot_churn_by_price_sensitivity(df)
plt.show()

**Finding:** Price sensitivity shows <2% difference in churn rate — not a meaningful differentiator. Discounts alone won't solve churn.

---

## 3. XGBoost Model Training

In [12]:
from src.model_training import train_xgboost, get_top_features

results = train_xgboost(df)

print("=== XGBoost Model Metrics ===")
for metric, value in results["metrics"].items():
    print(f"  {metric:>10s}: {value:.4f}")

print(f"\n=== Model Parameters ===")
for k, v in results["params"].items():
    print(f"  {k}: {v}")

=== XGBoost Model Metrics ===
    accuracy: 0.9301
   precision: 0.9266
      recall: 0.9458
          f1: 0.9361
         auc: 0.9754

=== Model Parameters ===
  n_estimators: 300
  max_depth: 6
  learning_rate: 0.1
  subsample: 0.8
  colsample_bytree: 0.8
  eval_metric: logloss
  random_state: 42
  scale_pos_weight: 0.849


### 3.1 Feature Importances

In [13]:
top_feat = get_top_features(results["feature_importances"], top_n=15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_feat["feature"][::-1], top_feat["importance_pct"][::-1],
        color=["#e74c3c" if v > 10 else "#3498db" for v in top_feat["importance_pct"][::-1]],
        edgecolor="white")
ax.set_xlabel("Importance (%)")
ax.set_title("XGBoost Feature Importances (Top 15)")
for i, (v, name) in enumerate(zip(top_feat["importance_pct"][::-1], top_feat["feature"][::-1])):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
fig.tight_layout()
plt.show()

print("\nTop 10 features:")
get_top_features(results["feature_importances"], top_n=10)


Top 10 features:


,feature,importance,importance_pct
0,membership_category,0.431221,43.099998
1,points_in_wallet,0.239588,24.000000
2,feedback,0.063331,6.300000
3,avg_transaction_value,0.021319,2.100000
4,tenure_days,0.013713,1.400000
5,avg_time_spent,0.013657,1.400000
6,avg_frequency_login_days,0.013323,1.300000
7,age,0.013174,1.300000
8,days_since_last_login,0.013114,1.300000
9,past_complaint,0.012732,1.300000


### 3.2 Confusion Matrix

In [14]:
cm = results["confusion_matrix"]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
fig.tight_layout()
plt.show()

print("\nClassification Report:")
print(results["classification_report"])


Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      3396
           1       0.93      0.95      0.94      4003

    accuracy                           0.93      7399
   macro avg       0.93      0.93      0.93      7399
weighted avg       0.93      0.93      0.93      7399



### 3.3 ROC Curve

In [15]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(results["y_test"], results["y_proba"])
auc_val = results["metrics"]["auc"]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#e74c3c", lw=2, label=f"XGBoost (AUC = {auc_val:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

---

## 4. Summary

### EDA Key Takeaways
1. **Membership category** is the dominant churn driver — two tiers have ~97% churn, two have 0%
2. **Feedback type** is a near-perfect churn signal — certain types have 0% churn, others ~65%
3. **Monetary engagement** (transaction value, wallet points) separates churners from retained
4. **Demographics, recency, tenure** are NOT useful churn predictors
5. `support_risk` and `past_complaint` are redundant (r=1.0)

### Model Results
- **XGBoost AUC: 97.6%** — confirms EDA findings quantitatively
- `membership_category` (46.4%) + `points_in_wallet` (21.0%) = **67% of model decisions**
- These insights feed directly into the **agentic retention pipeline** where AI agents generate, debate, and refine actionable retention strategies

### Next Step
The trained model powers the Analyst agent in the agentic pipeline, which identifies at-risk customers and their root causes. The Strategist and Critic agents then debate retention strategies until an actionable, measurable plan is approved.